In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('plot_paper')

enss=['b','c','d','e']
stouts=[5,7,10,13,15,20]
stout_chosen=10

tfphys=[0.4,0.5,0.6,0.7,0.8,0.9,1.0]
tcphys=[0.05,0.1,0.15,0.2,0.25,0.3]
tftcphys=[(tfphy,tcphy) for tfphy in tfphys for tcphy in tcphys if tfphy>=2*tcphy]

ens2Njk={'b':725,'c':400,'d':493,'e':516}

js_conn=['j+;conn','j-;conn']
js_discq=['j+;disc','js;disc','jc;disc'] 
js_gluon=[f'jg;stout{nst}' for nst in stouts]
js_disc=js_discq+js_gluon

In [2]:
enss=['b','c','d']
ens2q2gA={
    'b':{'ju':[0.867,0.018],'jd':[-0.408,0.014],'js':[-0.0262,0.0086],'jc':[0.0003,0.0045],'jq':[0.433,0.030],'j-':[1.275,0.023],'j+;conn':[0.585,0.013]},
    'c':{'ju':[0.832,0.022],'jd':[-0.415,0.021],'js':[-0.036,0.017],'jc':[-0.012,0.012],'jq':[0.368,0.065],'j-':[1.247,0.015],'j+;conn':[0.567,0.011]},
    'd':{'ju':[0.843,0.016],'jd':[-0.415,0.015],'js':[-0.034,0.013],'jc':[0.0076,0.0094],'jq':[0.401,0.049],'j-':[1.258,0.014],'j+;conn':[0.5659,0.0072]}
}
ens2rb={ens:{} for ens in enss}
for ens in enss:
    for j in ['jq','ju','jd','js','jc','j+;conn','j-']:
        ens2rb[ens][f'{j}']=yu.jackknife_pseudo([ens2q2gA[ens][j][0]],np.array([[ens2q2gA[ens][j][1]**2+1e-10]]),ens2Njk[ens])[:,0]
    ens2rb[ens][f'ju;conn']=(ens2rb[ens][f'j+;conn']+ens2rb[ens][f'j-'])/2
    ens2rb[ens][f'jd;conn']=(ens2rb[ens][f'j+;conn']-ens2rb[ens][f'j-'])/2
    ens2rb[ens][f'jq;conn']=ens2rb[ens][f'j+;conn']
    
ens2rb['a=#_MA']={}; ens2rb['a=#_const']={}; ens2rb['a=#_linear']={}

a2s_plt=yum.lat_a2s_plt
for j in ['jq','ju','jd','js','jc','ju;conn','jd;conn','jq;conn']:
    a2s=np.array([yu.ens2a[ens]**2 for ens in enss])
    y_jk=yu.superjackknife([ens2rb[ens][f'{j}'][:,None] for ens in enss])
        
    fits=[]
    
    def fitfunc(pars):
        g=pars
        return g+0*a2s
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,[1])
    def fitfunc_plt(pars):
        g=pars
        return g+0*a2s_plt
    pars_plt=yu.jackmap(fitfunc_plt,pars_jk)
    ens2rb['a=#_const'][f'{j}']=pars_plt
    fits.append(['const',pars_plt,chi2_jk,Ndof])
    
    def fitfunc(pars):
        g,c=pars
        return g+c*a2s
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,[1,1])
    def fitfunc_plt(pars):
        g,c=pars
        return g+c*a2s_plt
    pars_plt=yu.jackmap(fitfunc_plt,pars_jk)
    ens2rb['a=#_linear'][f'{j}']=pars_plt
    fits.append(['linear',pars_plt,chi2_jk,Ndof])
    
    pars_plt,props_jk=yu.jackMA(fits,systematicQ=True)
    ens2rb['a=#_MA'][f'{j}']=pars_plt

# for ft in ['const','linear','MA']:
#     ens2rb[f'a=#_{ft}'][f'jtot;{stout}']=ens2rb[f'a=#_{ft}'][f'jq;{stout}']+ens2rb[f'a=#_{ft}'][f'jg;{stout}']
#     ens2rb[f'a=#_{ft}'][f'jtot;conn;{stout}']=ens2rb[f'a=#_{ft}'][f'jq;conn;{stout}']

#     for fla in fla2iso.keys():
#         ens2rb[f'a=#_{ft}'][f'j{fla};{stout}']=np.sum([factor*ens2rb[f'a=#_{ft}'][f'j{iso};{stout}'] for factor,iso in fla2iso[fla]],axis=0)
#     for fla in fla2iso_conn.keys():
#         ens2rb[f'a=#_{ft}'][f'j{fla};conn;{stout}']=np.sum([factor*ens2rb[f'a=#_{ft}'][f'j{iso};conn;{stout}'] for factor,iso in fla2iso_conn[fla]],axis=0)

key2phy={}
for ens in ens2rb.keys():
    for j in ens2rb[ens].keys():
        t=ens2rb[ens][j]
        if ens not in enss:
            m,e=yu.jackme(t)
            t=yu.jackknife_pseudo(m,e,sum([ens2Njk[ens] for ens in ['b','c','d','e']]))
        key2phy[(ens,j)]=t
key2phy_gA=key2phy
enss=['b','c','d','e']

In [3]:
key2phy_A20 = yu.load_pkl_reg("key2phy_A20", pathlabel="analysis_xJ")
key2phy_B20 = yu.load_pkl_reg("key2phy_B20", pathlabel="analysis_xJ")
key2phy_J = {key:(key2phy_A20[key]+key2phy_B20[key])/2 for key in key2phy_A20.keys()}

key2phy_DeltaSigmaBy2={key:key2phy_gA[key]/2 for key in key2phy_gA.keys()}
key2phy_L={key:key2phy_J[key]-key2phy_DeltaSigmaBy2[key] for key in key2phy_DeltaSigmaBy2.keys() if key[1] not in ['j+;conn','j-']}

for j,j1 in zip(['jtot','jtot;conn'],['jq','jq;conn']):
    ens='a=#_MA'
    key2phy_DeltaSigmaBy2[(ens,j)]=key2phy_DeltaSigmaBy2[(ens,j1)]
    key2phy_L[(ens,j)]=key2phy_L[(ens,j1)]

# a2dep

In [4]:
def plot_A20_B20_J(which):
    key2phy = globals()[f"key2phy_{which}"]

    sty = {"jtot": ("gray", "o"), "jq": ("purple", "d"), "jg": ("cyan", "s"),
           "ju": ("red", "^"), "jd": ("green", "v"), "js": ("blue", "<"), "jc": ("orange", ">")}

    if which == "A20":
        rows = [(["jtot", "jq", "jg"], r"$\langle x\rangle_{q,g}$", (0.20, 1.40), [0.5, 1.0], 1.0),
                (["ju", "jd", "js", "jc"], r"$\langle x\rangle_q$", (-0.10, 0.58), [0.0, 0.2, 0.4], 0.0)]
        ratios, figsize = [1.15, 1.0], (5.6, 7.4)
        lab = dict(jtot=r"$\langle x\rangle_N$", jq=r"$\langle x\rangle_q$", jg=r"$\langle x\rangle_g$",
                   ju=r"$\langle x\rangle_u$", jd=r"$\langle x\rangle_d$",
                   js=r"$\langle x\rangle_s$", jc=r"$\langle x\rangle_c$")
    elif which == "B20":
        rows = [(["jtot"], r"$B_{20}^N$", (-0.30, 0.29), [-0.2, 0.0, 0.2], 0.0),
                (["jq", "jg"], r"$B_{20}^{q,g}$", (-0.30, 0.29), [-0.2, 0.0, 0.2], 0.0),
                (["ju", "jd", "js", "jc"], r"$B_{20}^q$", (-0.18, 0.18), [-0.1, 0.0, 0.1], 0.0)]
        ratios, figsize = [1.0, 1.0, 2.0], (5.6, 7.4)
        lab = {j: rf"$B_{{20}}^{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q", "g", "u", "d", "s", "c"])}
    else:
        rows = [(["jtot", "jq", "jg"], r"$J_{q,g}$", (0.08, 0.70), [0.2, 0.5], 0.5),
                (["ju", "jd", "js", "jc"], r"$J_q$", (-0.03, 0.28), [0.0, 0.1, 0.2], 0.0)]
        ratios, figsize = [1.15, 1.0], (5.6, 7.4)
        lab = {j: rf"$J_{{{s}}}$" for j, s in
               zip(["jtot", "jq", "jg", "ju", "jd", "js", "jc"], ["N", "q", "g", "u", "d", "s", "c"])}

    fig, axs = plt.subplots(len(rows), 1, figsize=figsize, sharex=True,
                            gridspec_kw={"height_ratios": ratios, "hspace": 0})
    axs = np.atleast_1d(axs)

    for ax, (js, ylabel, ylim, yticks, ref) in zip(axs, rows):
        for j in js:
            c, m = sty[j]
            x = np.asarray(yum.lat_a2s_plt)
            y, e = map(np.asarray, yu.jackme(key2phy[("a=#_MA", j)]))

            ax.plot(x, y, "--", color=c, lw=2)
            ax.fill_between(x, y - e, y + e, color=c, alpha=0.12)

            xs, ys, es = zip(*[(yu.ens2a[ens]**2, *yu.jackme(key2phy[(ens, j)])) for ens in enss])
            ax.errorbar(xs, ys, yerr=es, fmt=m, color=c, label=lab[j], capsize=6, lw=2, ms=6)

        ax.axhline(ref, color="black", ls=":", lw=2, marker="")
        ax.set(ylabel=ylabel, ylim=ylim, yticks=yticks)
        ax.tick_params(direction="in", top=True, right=True)

        if which=='B20' and j=='jtot':
            pass
        else:
            ax.legend(loc="upper right", frameon=True, ncol=len(js),
                  fontsize=15, handlelength=0.9, columnspacing=0.45,
                  handletextpad=0.25, borderpad=0.25)

    axs[-1].set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.008),
                xticks=[0.000, 0.003, 0.006])

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'a2dep_{t}')
    return fig, axs


for which in ["A20", "B20", "J"]:
    plot_A20_B20_J(which)

In [5]:
js=['jtot','jq','jv1','jv2','jv3','jg','ju','jd','js','jc']
index=[r'$\braket{x}'+(rf'_{{{j[1:]}}}$' if j not in ['jtot'] else r'_{N}$' ) for j in js]
columns=[yu.ens2label[ens] for ens in enss] + [r'$a=0$']

df=[[yu.jackme_un2str(key2phy_A20[(ens,j)]) for ens in enss] + [yu.jackme_un2str(key2phy_A20[('a=#_MA',j)][:,0])]  for j in js]
df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & B64 & C80 & D96 & E112 & $a=0$ \\
\hline
$\braket{x}_{N}$ & 0.993(64) & 0.906(67) & 1.05(11) & 1.24(15) & 1.079(91) \\
$\braket{x}_{q}$ & 0.594(52) & 0.495(50) & 0.735(85) & 0.85(12) & 0.709(79) \\
$\braket{x}_{v1}$ & 0.168(13) & 0.159(14) & 0.169(13) & 0.1717(96) & 0.1695(73) \\
$\braket{x}_{v2}$ & 0.482(39) & 0.451(25) & 0.524(24) & 0.399(44) & 0.481(20) \\
$\braket{x}_{v3}$ & 0.565(41) & 0.534(26) & 0.596(27) & 0.480(49) & 0.555(22) \\
$\braket{x}_{g}$ & 0.398(28) & 0.411(38) & 0.316(55) & 0.386(72) & 0.370(32) \\
$\braket{x}_{u}$ & 0.360(24) & 0.323(21) & 0.405(27) & 0.406(38) & 0.389(23) \\
$\braket{x}_{d}$ & 0.192(20) & 0.164(15) & 0.236(23) & 0.234(36) & 0.219(22) \\
$\braket{x}_{s}$ & 0.0351(90) & 0.018(12) & 0.059(21) & 0.121(30) & 0.063(19) \\
$\braket{x}_{c}$ & 0.0073(71) & -0.0096(96) & 0.035(18) & 0.093(27) & 0.039(17) \\
\end{tabular}



In [6]:
js=['jtot','jq','jv1','jv2','jv3','jg','ju','jd','js','jc']
index=[r'$B_{20}'+(rf'^{{{j[1:]}}}$' if j not in ['jtot'] else r'^{N}$' ) for j in js]
columns=[yu.ens2label[ens] for ens in enss] + [r'$a=0$']

df=[[yu.jackme_un2str(key2phy_B20[(ens,j)]) for ens in enss] + [yu.jackme_un2str(key2phy_B20[('a=#_MA',j)][:,0])]  for j in js]
df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & B64 & C80 & D96 & E112 & $a=0$ \\
\hline
$B_{20}^{N}$ & -0.090(63) & -0.036(68) & -0.03(12) & 0.02(12) & 0.021(68) \\
$B_{20}^{q}$ & -0.054(42) & -0.026(50) & -0.012(85) & -0.07(10) & -0.035(42) \\
$B_{20}^{v1}$ & 0.134(37) & 0.148(36) & 0.181(29) & 0.187(23) & 0.197(24) \\
$B_{20}^{v2}$ & -0.021(18) & -0.030(23) & -0.042(30) & -0.039(58) & -0.038(20) \\
$B_{20}^{v3}$ & 0.000(26) & -0.007(23) & -0.016(32) & -0.126(69) & -0.035(27) \\
$B_{20}^{g}$ & -0.035(34) & -0.009(37) & -0.016(63) & 0.093(60) & 0.055(46) \\
$B_{20}^{u}$ & 0.050(24) & 0.062(25) & 0.079(29) & 0.058(35) & 0.080(18) \\
$B_{20}^{d}$ & -0.084(21) & -0.086(21) & -0.102(26) & -0.129(35) & -0.116(17) \\
$B_{20}^{s}$ & -0.007(11) & 0.003(14) & 0.010(24) & -0.016(28) & 0.001(12) \\
$B_{20}^{c}$ & -0.0135(91) & -0.005(11) & 0.001(20) & 0.013(25) & 0.000(10) \\
\end{tabular}



In [7]:
js=['jtot','jq','jv1','jv2','jv3','jg','ju','jd','js','jc']
index=[r'$J'+(rf'_{{{j[1:]}}}$' if j not in ['jtot'] else r'_{N}$' ) for j in js]
columns=[yu.ens2label[ens] for ens in enss] + [r'$a=0$']

df=[[yu.jackme_un2str(key2phy_J[(ens,j)]) for ens in enss] + [yu.jackme_un2str(key2phy_J[('a=#_MA',j)][:,0])]  for j in js]
df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & B64 & C80 & D96 & E112 & $a=0$ \\
\hline
$J_{N}$ & 0.452(44) & 0.435(49) & 0.511(85) & 0.629(97) & 0.550(57) \\
$J_{q}$ & 0.270(33) & 0.234(36) & 0.361(63) & 0.390(79) & 0.337(45) \\
$J_{v1}$ & 0.151(23) & 0.153(24) & 0.175(21) & 0.179(15) & 0.183(15) \\
$J_{v2}$ & 0.231(21) & 0.211(18) & 0.241(19) & 0.180(36) & 0.222(14) \\
$J_{v3}$ & 0.282(25) & 0.263(19) & 0.290(21) & 0.177(42) & 0.260(18) \\
$J_{g}$ & 0.182(22) & 0.201(27) & 0.150(41) & 0.239(48) & 0.213(28) \\
$J_{u}$ & 0.205(19) & 0.192(20) & 0.242(22) & 0.232(26) & 0.234(16) \\
$J_{d}$ & 0.054(14) & 0.039(12) & 0.067(18) & 0.052(24) & 0.051(14) \\
$J_{s}$ & 0.0142(68) & 0.0103(93) & 0.034(17) & 0.052(21) & 0.032(12) \\
$J_{c}$ & -0.0031(56) & -0.0073(76) & 0.018(14) & 0.053(18) & 0.019(10) \\
\end{tabular}



# bar

In [8]:
def plot_bar(which):
    key2phy = globals()[f"key2phy_{which}"]

    colors = dict(ju="red", jd="green", js="blue", jc="orange",
                  jq="purple", jg="cyan", jtot="gray")
    names = dict(ju=r"$u$", jd=r"$d$", js=r"$s$", jc=r"$c$",
                 jq=r"$q$", jg=r"$g$", jtot=r"$\mathrm{Total}$")

    setup = {
        "A20": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
                r"$\langle x\rangle_{q,g}$", (0.0, 1.25),
                [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2], [1.0], 1.0, 2),
        "J": (["ju", "jd", "js", "jc", "jq", "jg", "jtot"],
              r"$J_{q,g}$", (0.0, 0.62),
              [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], [0.5], 0.5, 0),
        "DeltaSigmaBy2": (["ju", "jd", "js", "jc", "jtot"],
                          r"$\frac{1}{2}\Delta\Sigma_q$", (-0.32, 0.57),
                          [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
        "L": (["ju", "jd", "js", "jc", "jtot"],
              r"$L_{q,g}$", (-0.32, 0.57),
              [-0.25, 0.0, 0.25, 0.5], [0.0, 0.5], 0.5, -16),
    }

    js, ylabel, ylim, yticks, refs, norm, labelpad = setup[which]
    yr = ylim[1] - ylim[0]
    x = np.arange(len(js))

    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    for i, j in enumerate(js):
        c = colors[j]

        m, e = yu.jackme(key2phy[("a=#_MA", j)][:, 0])
        ax.bar(i, m, width=0.52, color=c, alpha=0.22, edgecolor=c, linewidth=1.3)
        ax.errorbar(i, m, yerr=e, fmt="none", color="black", capsize=6, lw=2)

        jc = f"{j};conn"
        if ("a=#_MA", jc) in key2phy:
            mc, _ = yu.jackme(key2phy[("a=#_MA", jc)][:, 0])
            ax.bar(i, mc, width=0.32, color=c, alpha=0.85, edgecolor=c, linewidth=1.3)

        txt = rf"${100*m/norm:.1f}({100*e/norm:.1f})\%$"
        ytxt = m + np.sign(m if m else 1) * (e + 0.035 * yr)
        ytxt = np.clip(ytxt, ylim[0] + 0.18 * yr, ylim[1] - 0.18 * yr)

        ax.text(i - 0.42, ytxt, txt, rotation=90,
                ha="center", va="center", fontsize=12, clip_on=True)

    for y in refs:
        ax.axhline(y, color="black", ls="--", lw=2, marker="")

    ax.set(
        ylabel=ylabel, ylim=ylim, yticks=yticks,
        xticks=x, xticklabels=[names[j] for j in js],
    )
    ax.set_xlim(-0.75, len(js) - 0.25)
    ax.yaxis.labelpad = labelpad
    ax.tick_params(direction="in", top=True, right=True)

    for s in ax.spines.values():
        s.set_linewidth(2)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'bar_{t}')
    return fig, ax

for which in ["A20", "J", "DeltaSigmaBy2", "L"]:
    plot_bar(which)

In [9]:
'''
# from Christos
sgm_piN=39.1(2.5)
sgm_s=36.7(6.7)
sgm_c=86(21)
sgm_sum=164(22)
'''
a=key2phy_A20[('a=#_MA','jq')][:,0]
ag=key2phy_A20[('a=#_MA','jg')][:,0]
Njk=len(a)
Mm=yu.jackknife_pseudo(0.164,0.022,Njk)[:,0] # from Christos above
MN=yu.jackknife_pseudo(0.9417,0.003,Njk)[:,0] # axial paper: 2309.05774
b=Mm/MN

noNumbersOnBar=False

def makePlot(case):
    def get(j):
        dic={'MN':MN, 'Mm':Mm, 'Mq':3*(a-b)/4*MN, 'Mg':3*ag/4*MN}
        
        if case==1:
            dic['Ma']=(1-b)/4*MN # = (dic['MN']-dic['Mm'])/4
        elif case==2:
            dic['Ma']=dic['MN']-dic['Mm']-dic['Mq']-dic['Mg']
        
        dic['M']=np.sum([dic[j] for j in ['Mm','Ma','Mq','Mg']],axis=0)
        return dic[j]/dic['MN']*100 
                

    fig, axs = yu.getFigAxs(1,1,Lrow=6,Lcol=8)

    js = ['Mm','Ma','Mq','Mg','M']
    # labels = [r'$u^+$', r'$d^+$', r'$s^+$', r'$c^+$', r'$\sum_{q^+ = u,d,s,c}$', r'$g$', 'Total']
    labels = [r'$M_m$', r'$M_a$', r'$M_q$', r'$M_g$', r'$M$']
    colors = ['b','orange','green','red','purple']

    ax=axs[0,0]
    ax.axhline(100,color='black',ls='--',marker='')
    # ax.axhline(0,color='black',ls='--',marker='')
    ax.set_ylabel(rf'% of M={yu.jackme_un2str(MN*1000)} MeV')

    x = np.arange(len(labels))
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlim([x[0]-3/4,x[-1]+3/4])
    ax.set_ylim([0,110])

    mes=[yu.jackme(get(j)) for j in js]

    factor=100
        
    bars=ax.bar(x, [ele[0] for ele in mes], yerr=[ele[1] for ele in mes], capsize=5, color=colors, alpha=0.8, width=0.5, edgecolor='grey')

    func=lambda res:'{:.1f}({:.1f})%'.format(res[0]*100/factor,res[1]*100/factor)
    percentages=[func(ele) for ele in mes]

    if not noNumbersOnBar:
        for i, (bar, pct) in enumerate(zip(bars, percentages)):
            height = bar.get_height()
            height = 1 if height < 20 else height - 20
            ax.text(bar.get_x() + 0.6, height, pct,
                    ha='center', va='bottom', fontsize=10, rotation=90)
            
    yu.finalizePlot(f'bar_mass_sumrule{case}')
    
makePlot(1)
makePlot(2)

In [10]:
js=['ju','jd','js','jc','jg','jtot']
whichs=['A20','J','DeltaSigmaBy2','L']

index=['u','d','s','c','g','Sum']
columns=[r'$\braket{x}$',r'$J$',r'$\frac{1}{2}\Delta\Sigma$',r'$L$']

def get(which,j):
    key2phy=globals()[f'key2phy_{which}']
    key=('a=#_MA',j)
    if key in key2phy:
        return yu.jackme_un2str(key2phy[key][:,0])
    return ''

df = [[get(which,j) for which in whichs] for j in js]

df=pd.DataFrame(df,index=index,columns=columns)

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
print(tex)

\begin{tabular}{cccccc}
 & $\braket{x}$ & $J$ & $\frac{1}{2}\Delta\Sigma$ & $L$ \\
\hline
u & 0.389(23) & 0.234(16) & 0.418(15) & -0.183(22) \\
d & 0.219(22) & 0.051(14) & -0.208(10) & 0.259(17) \\
s & 0.063(19) & 0.032(12) & -0.0171(86) & 0.049(14) \\
c & 0.039(17) & 0.019(10) & 0.0011(54) & 0.018(12) \\
g & 0.370(32) & 0.213(28) &  &  \\
Sum & 1.079(91) & 0.550(57) & 0.197(34) & 0.140(56) \\
\end{tabular}



# compare

In [11]:
pdflabels=['HERAPDF2.0','ABMP16','CT18','MSHT20','NNPDF4.0','PDF4LHC21','JAM22','CJ22']
pdfsets=['HERAPDF20_NNLO_EIG','ABMP16_4_nnlo','CT18NNLO','MSHT20nnlo_as118','NNPDF40_nnlo_as_01180','PDF4LHC21_40_pdfas','JAM22-PDF_proton_nlo','CJ22']
label2j2me_A20=yu.load_pkl_reg('label2j2me_avgx',pathlabel='lhapdf')
label2j2me_A20={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_A20[label].items()} for label in label2j2me_A20.keys()}

pdflabels_pol=['DSSV14','JAM22','NNPDFpol2.0']
pdfsets_pol=['DSSV14pol','JAM22-PPDF_proton_nlo','NNPDFpol20_nnlo_as_01180_mhou']
label2j2me_gA=yu.load_pkl_reg('label2j2me_gA_xmin=1e-3',pathlabel='lhapdf')
label2j2me_gA={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

label2j2me_DeltaSigmaBy2={label:{j:(m/2,e/2) for j,(m,e) in label2j2me_gA[label].items()} for label in label2j2me_gA.keys()}

In [12]:
def plot_compare(which):
    key2phy = globals()[f"key2phy_{which}"]
    get_src = getattr(yum, f"get_{which}_from_src")

    cfg = {
        "A20": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$\langle x\rangle_u$", r"$\langle x\rangle_d$", r"$\langle x\rangle_s$",
                    r"$\langle x\rangle_c$", r"$\langle x\rangle_g$"],
            phen=(["ABMP16", "CT18", "MSHT20", "NNPDF4.0", "JAM22", "CJ22"],
                  pdflabels, pdfsets, label2j2me_A20),
            srcs=[r"$\chi$QCD18", "MIT24", "ETM20"],
            xlim=[(0.22, 0.52), (0.06, 0.36), (-0.10, 0.20), (-0.12, 0.18), (0.10, 0.60)],
            xticks=[[0.30, 0.45], [0.15, 0.30], [0.0, 0.1], [0.0, 0.1], [0.25, 0.45]],
            figsize=(13.5, 3.5),
        ),
        "J": dict(
            js=["ju", "jd", "js", "jc", "jg"],
            xlabel=[r"$J_u$", r"$J_d$", r"$J_s$", r"$J_c$", r"$J_g$"],
            phen=None, srcs=["MIT24", "ETM20"],
            xlim=[(0.12, 0.32), (-0.04, 0.16), (-0.08, 0.12), (-0.09, 0.09), (0.10, 0.30)],
            xticks=[[0.18, 0.28], [0.0, 0.1], [0.0, 0.08], [0.0, 0.07], [0.15, 0.25]],
            figsize=(13.5, 1.1),
        ),
        
        "DeltaSigmaBy2": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$\frac{1}{2}\Delta\Sigma_u$", r"$\frac{1}{2}\Delta\Sigma_d$",
                    r"$\frac{1}{2}\Delta\Sigma_s$", r"$\frac{1}{2}\Delta\Sigma_c$"],
            phen=(["DSSV14", "JAM22", "NNPDFpol2.0"],
                  pdflabels_pol, pdfsets_pol, label2j2me_DeltaSigmaBy2),
            srcs=[r"$\chi$QCD18", "PNDME25", "Mainz26", "ETM20"],
            xlim=[(0.32, 0.52), (-0.31, -0.11), (-0.12, 0.10), (-0.09, 0.09)],
            xticks=[[0.35, 0.45], [-0.25, -0.15], [-0.08, 0.0], [-0.06, 0.0, 0.06]],
            figsize=(11.5, 2.8),
        ),
        "L": dict(
            js=["ju", "jd", "js", "jc"],
            xlabel=[r"$L_u$", r"$L_d$", r"$L_s$", r"$L_c$"],
            phen=None, srcs=["ETM20"],
            xlim=[(-0.30, -0.10), (0.16, 0.36), (-0.04, 0.16), (-0.09, 0.09)],
            xticks=[[-0.25, -0.15], [0.2, 0.3], [0.0, 0.1], [0.0, 0.07]],
            figsize=(11.5, 0.8),
        ),
    }[which]

    def missing(v):
        return v is None or (isinstance(v, tuple) and any(x is None for x in v))

    def split_err(e):
        if isinstance(e, tuple):
            estat, esys = e
            return estat, np.sqrt(estat**2 + esys**2)
        return None, e

    phen_rows = []
    if cfg["phen"] is not None:
        order, pdflabs, pdfsets_, label2j2me = cfg["phen"]
        lab2set = dict(zip(pdflabs, pdfsets_))
        phen_rows = [(lab, lab2set[lab]) for lab in order if lab in lab2set]

    ylabels = [lab for lab, _ in phen_rows] + cfg["srcs"] + ["This Work"]
    y = np.arange(len(ylabels))
    ymap = dict(zip(ylabels, y))

    fig, axs = plt.subplots(1, len(cfg["js"]), figsize=cfg["figsize"], sharey=True, gridspec_kw={"wspace": 0.05})
    axs = np.atleast_1d(axs)

    for ax, j, xl, xlim, xticks in zip(axs, cfg["js"], cfg["xlabel"], cfg["xlim"], cfg["xticks"]):
        m, e = yu.jackme(key2phy[("a=#_MA", j)][:, 0])
        ax.axvspan(m - e, m + e, color="red", alpha=0.20)
        # ax.axvline(m, color="red", alpha=0.45)
        ax.errorbar(m, ymap["This Work"], xerr=e, fmt="s", color="red")

        if cfg["phen"] is not None:
            label2j2me = cfg["phen"][3]
            for lab, label in phen_rows:
                if which == "DeltaSigmaBy2" and j == "jc" and lab in ["DSSV14", "JAM22"]:
                    continue
                val = label2j2me.get(label, {}).get(j)
                if missing(val):
                    continue
                mp, ep = val
                if missing((mp, ep)):
                    continue
                _, ep = split_err(ep)
                ax.errorbar(mp, ymap[lab], xerr=ep, fmt="^", color="black")

        for src in cfg["srcs"]:
            val = get_src(src, j)
            if missing(val):
                continue
            ms, es = val
            if missing((ms, es)):
                continue
            estat, etot = split_err(es)

            if src == "ETM20":
                col, mk, mfc = "green", "o", "white"
            elif src == "MIT24":
                col, mk, mfc = "blue", "d", "white"
            else:
                col, mk, mfc = "blue", "d", "blue"

            ax.errorbar(ms, ymap[src], xerr=etot, fmt=mk, color=col, mfc=mfc, mec=col)
            if estat is not None:
                ax.errorbar(ms, ymap[src], xerr=estat, fmt="none", color=col)

        ax.set(xlabel=xl, xlim=xlim, xticks=xticks)
        ax.set_ylim(len(ylabels) - 0.25, -0.75)
        ax.tick_params(direction="in", top=True, right=True, labelsize=16)
        for s in ax.spines.values():
            s.set_linewidth(2)

    axs[0].set_yticks(y)
    axs[0].set_yticklabels(ylabels, fontsize=18)
    for ax in axs[1:]:
        ax.tick_params(labelleft=False)

    t={'A20':'avgx'}[which] if which in ['A20'] else which
    yu.finalizePlot(f'compare_{t}',tightQ=False)
    return fig, axs


for which in ["A20", "J", "DeltaSigmaBy2", "L"]:
    plot_compare(which)